# 08 — Milestone 3 Plots

Generates the **six per-category donor co-movement panels** referenced in [`presentation/milestone3/milestone3.tex`](../presentation/milestone3/milestone3.tex).

For each of the six donor categories, a side-by-side figure shows Brent (black) against that category's donors over the **Hormuz** (left) and **Russia** (right) windows, rebased to 100 at each pre-window start. This answers Milestone 2 review comment 1 (SUTVA): the audience sees, category by category, how each donor group co-moves with Brent and how it behaves through `T0`.

Outputs → `plots/milestone3/donors_<key>_side_by_side.png`:

| Category | File |
|---|---|
| Metals | `donors_metals_side_by_side.png` |
| Agriculturals | `donors_ags_side_by_side.png` |
| Equities | `donors_equities_side_by_side.png` |
| FX | `donors_fx_side_by_side.png` |
| Rates/credit | `donors_rates_side_by_side.png` |
| Volatility | `donors_vol_side_by_side.png` |

Style mirrors the side-by-side panel in [`07_Milestone2_Plots.ipynb`](07_Milestone2_Plots.ipynb) (same `build_event_panel`, same windows). No model fits required — raw Brent + donor levels only.

**Donor categorization** uses the **19-donor shared pool** actually fed to the SCM ensemble (Metals 3, Agriculturals 3, Equities 2, FX 8, Rates/credit 2, Volatility 1), confirmed against `data/validation/donor_importance_consensus.csv`. This deliberately does **not** include Cotton or EM_Eq — the milestone2 deck's "21" figure was an overcount.

**This notebook now generates *all* milestone-3 figures** (19-donor shared pool):
the six per-category donor panels (below), plus the three slide figures —
`donor_importance_heatmap.png` (Comment 2), `naive_baselines.png` (Comment 3), and
`postwindow_sensitivity.png` (Summary). Re-run top-to-bottom to refresh the deck assets.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = ROOT / 'data'
PLOTS_DIR = ROOT / 'plots' / 'milestone3'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- data ----------
brent = (pd.read_csv(DATA / 'brent_spot.csv', parse_dates=['Date'])
           .rename(columns={'Date': 'date', 'Price': 'Brent'})
           .set_index('date').sort_index().dropna())
donors = pd.read_parquet(DATA / 'donors.parquet')

# ---------- donor categories: included (solid) vs excluded (dashed) ----------
# Included = the 18-donor shared/preferred pool fed to the SCM ensemble
# (confirmed against data/validation/donor_importance_consensus.csv).
# Excluded = candidates in the same category dropped by the SUTVA audit
# (oil-supply / disruption path) — shown dashed for visual contrast.
DONOR_CATEGORIES = {
    'Metals':       {'key': 'metals',   'included': ['Silver', 'Platinum', 'Gold'],
                                         'excluded': ['Copper', 'IronOre', 'Palladium']},
    'Agriculturals':{'key': 'ags',      'included': ['Coffee', 'Sugar', 'LiveCattle'],
                                         'excluded': ['Wheat', 'Corn', 'Soybeans', 'Cotton']},
    'Equities':     {'key': 'equities', 'included': ['SP500', 'Nikkei'],
                                         'excluded': ['EM_Eq']},
    'FX':           {'key': 'fx',       'included': ['AUD', 'JPY', 'CHF', 'CNY', 'INR', 'KRW', 'ZAR', 'MXN'],
                                         'excluded': ['EUR', 'GBP', 'DXY']},
    'Rates/credit': {'key': 'rates',    'included': ['TLT', 'HYG'],
                                         'excluded': ['US10Y']},
    'Volatility':   {'key': 'vol',      'included': [],
                                         'excluded': ['VIX']},
}
assert sum(len(c['included']) for c in DONOR_CATEGORIES.values()) == 18

# ---------- focal events (Hormuz left, Russia right) ----------
FOCAL_EVENTS = [
    {'name': 'Strait of Hormuz crisis', 'slug': 'hormuz',
     'T0': pd.Timestamp('2026-02-28'), 'pre_start': pd.Timestamp('2024-06-01'),
     'post_end': pd.Timestamp('2026-05-31')},
    {'name': 'Russia invades Ukraine', 'slug': 'russia',
     'T0': pd.Timestamp('2022-02-24'), 'pre_start': pd.Timestamp('2020-07-01'),
     'post_end': pd.Timestamp('2022-09-30')},
]

# 20-color palette so even FX (8 incl + 3 excl = 11 lines) stays distinguishable.
PALETTE = list(plt.get_cmap('tab20').colors)

print(f'Brent:  {brent.shape}, {brent.index.min().date()} -> {brent.index.max().date()}')
print(f'Donors: {donors.shape}')
print(f'Output: {PLOTS_DIR}')

Brent:  (9902, 1), 1987-05-20 -> 2026-06-01
Donors: (5511, 32)
Output: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3
Last run: 2026-06-10 10:40:26


In [2]:
def build_event_panel(event, cols):
    """Brent + selected donor cols, pre_start..post_end, rebased to 100 at first in-window date."""
    data = (brent[['Brent']].join(donors[cols], how='outer')
                            .sort_index()
                            .loc[event['pre_start']:event['post_end']])
    data = data.ffill(limit=5).dropna()
    if data.empty:
        return None
    norm = 100 * data.divide(data.iloc[0])
    norm['_days_from_T0'] = (norm.index - event['T0']).days
    return norm


def plot_category(label, cfg):
    inc, exc = cfg['included'], cfg['excluded']
    cols = inc + exc
    color_map = dict(zip(cols, PALETTE[:len(cols)]))
    # Larger figure — the graph is the slide highlight.
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.6), sharey=False)

    for ax, event in zip(axes, FOCAL_EVENTS):
        panel = build_event_panel(event, cols)
        if panel is None:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
            continue
        x = panel['_days_from_T0'].values
        # Included: solid, full opacity.
        for c in inc:
            ax.plot(x, panel[c].values, color=color_map[c], linewidth=1.8, alpha=0.9, label=c)
        # Excluded: dashed, dimmer, labelled "(excl.)".
        for c in exc:
            ax.plot(x, panel[c].values, color=color_map[c], linewidth=1.6, alpha=0.75,
                    linestyle='--', dashes=(4, 2), label=f'{c} (excl.)')
        ax.plot(x, panel['Brent'].values, color='black', linewidth=2.8, label='Brent', zorder=10)
        ax.axvline(x=0, color='black', linestyle=':', linewidth=1.4, alpha=0.7, zorder=8)
        ax.set_xlabel(f"Days from T0 ({event['T0'].date()})", fontsize=14)
        ax.set_title(f"{event['name']}\n{event['pre_start'].date()} -> {event['post_end'].date()}",
                     fontsize=15, pad=6)
        ax.grid(alpha=0.22, linestyle='-', linewidth=0.5)
        ax.tick_params(labelsize=12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ncol = 2 if len(cols) > 5 else 1
        ax.legend(loc='upper left', frameon=True, fontsize=11, framealpha=0.9, ncol=ncol)

    axes[0].set_ylabel('Index (pre-window start = 100)', fontsize=14)
    fig.suptitle(f'{label} — included (solid) vs excluded (dashed) donors vs Brent',
                 fontsize=17, y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out = PLOTS_DIR / f"donors_{cfg['key']}_side_by_side.png"
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out}  ({len(inc)} incl, {len(exc)} excl)")


for label, cfg in DONOR_CATEGORIES.items():
    plot_category(label, cfg)

print('\nAll six per-category panels written to', PLOTS_DIR)

Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donors_metals_side_by_side.png  (3 incl, 3 excl)


Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donors_ags_side_by_side.png  (3 incl, 4 excl)


Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donors_equities_side_by_side.png  (2 incl, 1 excl)
Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donors_fx_side_by_side.png  (8 incl, 3 excl)


Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donors_rates_side_by_side.png  (2 incl, 1 excl)
Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donors_vol_side_by_side.png  (0 incl, 1 excl)

All six per-category panels written to /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3
Last run: 2026-06-10 10:40:27


In [3]:
# ---- Headline ATT across post-event windows (for the summary slide) ----
# Source: data/validation/final_postwindow_sensitivity.csv (ensemble median + IQR per horizon).
pw = pd.read_csv(DATA / 'validation' / 'final_postwindow_sensitivity.csv')

fig, ax = plt.subplots(figsize=(11, 5.5))
styles = {
    'hormuz': dict(color='#8B0000', marker='o', label='Hormuz 2026'),
    'russia': dict(color='#1f77b4', marker='s', label='Russia 2022'),
}
for ev, st in styles.items():
    d = pw[pw.event == ev].sort_values('n_post')
    ax.fill_between(d.n_post, d.iqr_lo, d.iqr_hi, color=st['color'], alpha=0.13, zorder=1)
    ax.plot(d.n_post, d.ens_median_gap_pct, lw=2.6, marker=st['marker'], ms=8,
            color=st['color'], label=st['label'], zorder=5)
    # Merge labels for points closer than ~6 trading days (e.g. Hormuz 3m vs full).
    groups = []
    for _, r in d.iterrows():
        if groups and (r.n_post - groups[-1]['x']) < 6:
            groups[-1]['labels'].append(r.horizon)
            groups[-1]['x'], groups[-1]['y'] = r.n_post, r.ens_median_gap_pct
        else:
            groups.append({'x': r.n_post, 'y': r.ens_median_gap_pct, 'labels': [r.horizon]})
    for g in groups:
        ax.annotate(f"{'/'.join(g['labels'])}\n{g['y']:.0f}%", (g['x'], g['y']),
                    textcoords='offset points', xytext=(0, 12), fontsize=12, ha='center',
                    color=st['color'], fontweight='semibold')

ax.axhline(0, color='black', lw=0.7, alpha=0.5)
ax.set_xlabel('Post-event window length (trading days from $T_0$)', fontsize=14)
ax.set_ylabel('Ensemble-median ATT (%)', fontsize=14)
ax.set_title('Headline ATT across post-event windows  (ensemble median, shaded = IQR)', fontsize=16)
ax.set_ylim(0, 70)
ax.grid(alpha=0.25, linewidth=0.5)
ax.tick_params(labelsize=12.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=13, loc='upper right', frameon=True, framealpha=0.9)
fig.tight_layout()

out = PLOTS_DIR / 'postwindow_sensitivity.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print('Saved:', out)

Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/postwindow_sensitivity.png
Last run: 2026-06-10 10:40:27


In [4]:
# ---- Comment 2: donor-importance heatmap (blue scale, gray = not used, category-grouped) ----
# -> plots/milestone3/donor_importance_heatmap.png
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from lib.config import DONOR_POOL_VARIANT
from lib.data import load_fit

HM_MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
HM_MODEL_DISPLAY = {'convex_scm': 'Convex SCM', 'ascm': 'ASCM', 'elastic_net': 'Elastic-net',
                    'xgboost': 'XGBoost', 'bayesian_ridge': 'Bayesian\nRidge'}
# rows grouped by category (19-donor shared pool; VIX included)
HM_CATS = {'Metals': ['Silver', 'Platinum', 'Gold'],
           'Agriculturals': ['Coffee', 'Sugar', 'LiveCattle'],
           'Equities': ['SP500', 'Nikkei'],
           'FX': ['AUD', 'JPY', 'CHF', 'CNY', 'INR', 'KRW', 'ZAR', 'MXN'],
           'Rates/credit': ['TLT', 'HYG'],
           'Volatility': ['VIX']}
HM_CAT_ABBR = {'Metals': 'Metals', 'Agriculturals': 'Agric.', 'Equities': 'Equity',
               'FX': 'FX', 'Rates/credit': 'Rates', 'Volatility': 'Vol.'}

hm_ordered, hm_bounds, hm_blocks = [], [], []
for cat, mem in HM_CATS.items():
    if hm_ordered:
        hm_bounds.append(len(hm_ordered))
    s0 = len(hm_ordered); hm_ordered.extend(mem); hm_blocks.append((cat, s0, len(hm_ordered)))


def _hm_importance(fit):
    w = fit['weights'].reindex(hm_ordered).fillna(0).abs(); t = w.sum()
    return (w / t if t > 0 else w).values


hm_mats = {}
for ev in FOCAL_EVENTS:
    M = np.zeros((len(hm_ordered), len(HM_MODELS)))
    for j, m in enumerate(HM_MODELS):
        f = load_fit(ev['slug'], 'preferred', m, variant=DONOR_POOL_VARIANT)
        if f is not None:
            M[:, j] = _hm_importance(f)
    hm_mats[ev['slug']] = M
hm_vmax = max(M.max() for M in hm_mats.values())

# sequential blue for used donors (truncated so small weights stay visibly blue);
# zeros masked -> flat light gray = "not used".
hm_cmap = LinearSegmentedColormap.from_list('b', plt.cm.Blues(np.linspace(0.22, 1.0, 256)))
hm_cmap.set_bad('#e0e0e0')

fig, axes = plt.subplots(1, 2, figsize=(14, 9))
fig.subplots_adjust(left=0.10, right=0.985, top=0.95, bottom=0.13, wspace=0.05)
im = None
for ax, ev in zip(axes, FOCAL_EVENTS):
    M = hm_mats[ev['slug']]; Mm = np.ma.masked_where(M == 0, M)
    im = ax.imshow(Mm, cmap=hm_cmap, aspect='auto', vmin=0, vmax=hm_vmax)
    ax.set_xticks(np.arange(-.5, len(HM_MODELS), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(hm_ordered), 1), minor=True)
    ax.grid(which='minor', color='#c8c8c8', linewidth=0.6); ax.tick_params(which='minor', length=0)
    ax.set_xticks(range(len(HM_MODELS))); ax.set_xticklabels([HM_MODEL_DISPLAY[m] for m in HM_MODELS], fontsize=13)
    ax.set_yticks(range(len(hm_ordered))); ax.set_yticklabels(hm_ordered if ax is axes[0] else [], fontsize=12.5)
    for b in hm_bounds:
        ax.axhline(b - 0.5, color='black', linewidth=1.1)
    for i in range(len(hm_ordered)):
        for j in range(len(HM_MODELS)):
            v = M[i, j]
            if v > 0.04:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10.5,
                        color='white' if v > 0.55 * hm_vmax else 'black')
    ax.set_title(ev['name'], fontsize=17, pad=8)
    for sp in ax.spines.values():
        sp.set_visible(False)

for cat, s, e in hm_blocks:                                    # category labels at far left
    axes[0].text(-1.9, (s + e - 1) / 2, HM_CAT_ABBR[cat], rotation=90, va='center', ha='center',
                 fontsize=12, fontweight='bold', color='#555', clip_on=False)

cax = fig.add_axes([0.30, 0.06, 0.34, 0.018])                  # horizontal colorbar along the bottom
cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
cbar.set_label('within-model importance share', fontsize=13); cbar.ax.tick_params(labelsize=11.5)
fig.legend(handles=[Patch(facecolor='#e0e0e0', edgecolor='#c8c8c8', label='not used (weight 0)')],
           loc='center left', bbox_to_anchor=(0.70, 0.075), fontsize=13, frameon=False)
out = PLOTS_DIR / 'donor_importance_heatmap.png'
fig.savefig(out, dpi=200, bbox_inches='tight'); plt.close(fig)
print('Saved:', out)

Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/donor_importance_heatmap.png
Last run: 2026-06-10 10:40:27


In [5]:
# ---- Comment 3: naive counterfactual baselines (observed vs SCM synthetic vs naive) ----
# Faithful static copy of 06_Ensemble_Final cells 15-16, over the ENTIRE pre+post window.
# Legend inside the left panel (matches the per-model synthetic-paths appendix figure).
# -> plots/milestone3/naive_baselines.png
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.dates as mdates
from lib.config import DONOR_POOL_VARIANT
from lib.data import build_panel, load_fit

NB_MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
ARIMA_HALFLIFE_BD = 126


def naive_counterfactuals(logbrent, t0):
    """Univariate counterfactuals on log-Brent over the post-window (06 cell 15)."""
    pre = logbrent[logbrent.index < t0]; post = logbrent[logbrent.index >= t0]
    n_pre, n_post = len(pre), len(post); last = pre.iloc[-1]
    cf = {'rw_flat': pd.Series(last, index=post.index)}
    drift = pre.diff().dropna().mean()
    cf['rw_drift'] = pd.Series(last + drift * np.arange(1, n_post + 1), index=post.index)
    slope, intercept = np.polyfit(np.arange(n_pre), pre.values, 1)
    cf['lin_trend'] = pd.Series(intercept + slope * np.arange(n_pre, n_pre + n_post), index=post.index)
    tr = 'n' if slope > 0 else 'c'
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            res = SARIMAX(pre.values, order=(1, 0, 0), trend=tr, enforce_stationarity=True,
                          enforce_invertibility=False).fit(disp=0, maxiter=300)
        cf['arima_rw'] = pd.Series(np.asarray(res.get_forecast(steps=n_post).predicted_mean), index=post.index)
    except Exception:
        cf['arima_rw'] = cf['rw_flat'].copy()
    mean_log = pre.mean(); phi_mr = 0.5 ** (1.0 / ARIMA_HALFLIFE_BD)
    cf['arima_mr'] = pd.Series(mean_log + (last - mean_log) * phi_mr ** np.arange(1, n_post + 1), index=post.index)
    return cf


NB_STYLE = [('lin_trend', '#2ca02c', 'linear pre-trend', '-.'),
            ('rw_drift',  '#9467bd', 'RW + drift', ':'),
            ('rw_flat',   '#ff7f0e', 'ARIMA(0,1,0)', '--'),
            ('arima_rw',  '#8c564b', 'ARIMA(1,0,0)', (0, (5, 1))),
            ('arima_mr',  '#e377c2', 'ARIMA mean-reverting', (0, (3, 1, 1, 1)))]

fig, axes = plt.subplots(1, 2, figsize=(15, 6.0))
for ax, ev in zip(axes, FOCAL_EVENTS):
    panel, meta = build_panel(event=ev['slug'], window='preferred', variant=DONOR_POOL_VARIANT)
    t0 = meta['t0']; obs_log = panel['Brent']                  # entire pre+post window
    cf = naive_counterfactuals(obs_log, t0)
    fits = [load_fit(ev['slug'], 'preferred', m, variant=DONOR_POOL_VARIANT) for m in NB_MODELS]
    synth = pd.concat([np.exp(f['synth']) for f in fits if f is not None], axis=1).mean(axis=1)
    ax.plot(obs_log.index, np.exp(obs_log).values, color='black', lw=2.4, label='Observed Brent', zorder=6)
    ax.plot(synth.index, synth.values, color='#8B0000', lw=1.9, ls='--', label='SCM synthetic (ens. mean)', zorder=5)
    for nm, color, lbl, dash in NB_STYLE:
        ax.plot(cf[nm].index, np.exp(cf[nm]).values, color=color, lw=1.4, ls=dash, label=lbl)
    ax.axvline(t0, color='grey', lw=1.2, ls=':')
    ax.set_title(ev['name'], fontsize=16, pad=8)
    ax.grid(alpha=0.25, lw=0.5); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.tick_params(labelsize=12.5)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

axes[0].set_ylabel('Brent (USD / bbl)', fontsize=14)
axes[0].legend(loc='upper left', fontsize=11, framealpha=0.92)   # legend inside the left panel
fig.tight_layout()
out = PLOTS_DIR / 'naive_baselines.png'
fig.savefig(out, dpi=200, bbox_inches='tight'); plt.close(fig)
print('Saved:', out)

Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/naive_baselines.png
Last run: 2026-06-10 10:40:28


In [6]:
# ---- Appendix: observed Brent vs each model's synthetic counterfactual (per event) ----
# -> plots/milestone3/model_paths.png
import matplotlib.dates as mdates
from lib.config import DONOR_POOL_VARIANT
from lib.data import load_fit

MP_MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
MP_DISPLAY = {'convex_scm': 'Convex SCM', 'ascm': 'ASCM', 'elastic_net': 'Elastic-net',
              'xgboost': 'XGBoost', 'bayesian_ridge': 'Bayesian Ridge'}
MP_COLORS = {'convex_scm': '#1f77b4', 'ascm': '#ff7f0e', 'elastic_net': '#2ca02c',
             'xgboost': '#d62728', 'bayesian_ridge': '#9467bd'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6.0))
for ax, ev in zip(axes, FOCAL_EVENTS):
    fits = {m: load_fit(ev['slug'], 'preferred', m, variant=DONOR_POOL_VARIANT) for m in MP_MODELS}
    fits = {m: f for m, f in fits.items() if f is not None}
    any_fit = next(iter(fits.values()))
    t_start = any_fit['t_pre_start']
    t_end = max(f['gap'].index.max() for f in fits.values())

    obs = brent.loc[t_start:t_end, 'Brent']
    ax.plot(obs.index, obs.values, color='black', lw=2.2, label='Observed Brent', zorder=10)
    for m, f in fits.items():
        s = np.exp(f['synth'])                                 # log -> USD/bbl
        ax.plot(s.index, s.values, color=MP_COLORS[m], lw=1.3, alpha=0.9, label=MP_DISPLAY[m])
    ax.axvline(ev['T0'], color='black', ls='--', lw=1.3, alpha=0.7, zorder=8)
    ax.set_title(ev['name'], fontsize=16, pad=8)
    ax.set_xlabel('Date', fontsize=13.5)
    ax.grid(alpha=0.22, lw=0.5)
    ax.tick_params(labelsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

axes[0].set_ylabel('Brent (USD / bbl)', fontsize=13.5)
axes[0].legend(loc='upper left', fontsize=12, framealpha=0.92)
fig.suptitle('Observed Brent vs per-model synthetic counterfactuals (19-donor pool)', fontsize=16, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.97])
out = PLOTS_DIR / 'model_paths.png'
fig.savefig(out, dpi=200, bbox_inches='tight'); plt.close(fig)
print('Saved:', out)

Saved: /Users/cherrylchico/Desktop/BSE/T3/Thesis/Brent_Analysis/plots/milestone3/model_paths.png
Last run: 2026-06-10 10:40:29
